# GPU Liquid Cooling — Interactive Sizing Notebook

This notebook walks through three core thermal engineering questions at H100/B200 scale using `thermal-mcp-server` — the same physics engine that powers the MCP tools.

**Sections:**
1. NVL72 CDU sizing — the procurement question
2. Series vs. parallel rack topology — what actually changes
3. Flow rate optimization and thermal margin

Each section starts with the engineering question, shows the physics, and interprets the result.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardovietri/thermal-mcp-server/blob/main/examples/interactive_sizing.ipynb)

In [ ]:
# Install if running in Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'thermal-mcp-server', 'matplotlib'], check=True, capture_output=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from thermal_mcp_server.physics import analyze, analyze_rack, optimize_flow
from thermal_mcp_server.schemas import (
    AnalyzeColdplateInput, AnalyzeRackInput, OptimizeFlowRateInput, Geometry
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# B200-specific cold plate geometry
# 60 channels x 0.7 mm x 1.5 mm, 100 mm long, 160 cm2 contact area
# Engineering estimate — NVIDIA does not publish cold plate geometry
B200_PLATE = Geometry(
    channel_count=60, channel_width_m=0.7e-3, channel_height_m=1.5e-3,
    channel_length_m=0.10, base_thickness_m=1.5e-3,
    contact_area_m2=0.016, copper_k_w_mk=385.0,
)
B200_R_JC = 0.02   # K/W — engineering estimate
B200_R_TIM = 0.015  # K/W

print("thermal-mcp-server loaded successfully")

---
## Section 1: NVL72 CDU Sizing — The Procurement Question

### The problem

You're deploying an NVL72: 72 B200 GPUs at ~1,200 W each, 86.4 kW rack TDP. You need to spec a CDU. The vendor (Vertiv, CoolIT, Motivair) needs:

1. Total flow rate (LPM)
2. Heat rejection capacity (kW)
3. Maximum cold plate ΔP at that flow (bar) — drives CDU pump spec
4. Return water temperature — drives facility chilled water design

The CDU vendor will ask "what flow do you need?" You need a number, not a guess.

### The physics

Junction temperature is:
```
Tj = T_inlet + Q × (R_jc + R_tim + R_base + R_conv) + ΔT_coolant / 2
```
- R_conv = 1 / (h × A_contact) decreases with flow (higher velocity → higher Re → higher Nu → higher h)
- ΔT_coolant = Q / (ṁ × cp) also decreases with flow
- But pump power ∝ Q³ — so more flow costs a lot

The optimization finds the minimum Q where Tj ≤ Tj_limit.

In [ ]:
# Minimum flow per GPU to keep B200 below 75°C at 25°C CDU supply
B200_TDP_W = 1200.0
B200_TJ_LIMIT_C = 75.0   # SemiAnalysis estimate; NVIDIA does not publish
NVL72_GPU_COUNT = 72
CDU_SUPPLY_C = 25.0

opt_flow_lpm, opt_analysis = optimize_flow(OptimizeFlowRateInput(
    heat_load_w=B200_TDP_W,
    max_junction_temp_c=B200_TJ_LIMIT_C,
    inlet_temp_c=CDU_SUPPLY_C,
    coolant="water",
    r_jc_k_per_w=B200_R_JC,
    r_tim_k_per_w=B200_R_TIM,
    geometry=B200_PLATE,
))

total_flow_lpm = opt_flow_lpm * NVL72_GPU_COUNT

# Full rack spec at minimum flow
rack_min = analyze_rack(AnalyzeRackInput(
    gpu_count=NVL72_GPU_COUNT, topology="parallel",
    heat_load_per_gpu_w=B200_TDP_W, total_flow_lpm=total_flow_lpm,
    cdu_supply_temp_c=CDU_SUPPLY_C, coolant="water",
    r_jc_k_per_w=B200_R_JC, r_tim_k_per_w=B200_R_TIM, geometry=B200_PLATE,
))

print("NVL72 CDU Specification (at Tj = 75°C design limit):")
print(f"  Total flow:           {total_flow_lpm:.0f} LPM  ({opt_flow_lpm:.1f} LPM/GPU)")
print(f"  Heat rejection:       {rack_min.total_heat_load_w/1000:.1f} kW")
print(f"  Cold plate ΔP:        {rack_min.total_pressure_drop_pa/1e5:.3f} bar")
print(f"    (real system ΔP:    add 20–50% for manifold/headers)")
print(f"  CDU return temp:      {rack_min.cdu_outlet_temp_c:.1f}°C  (supply: {CDU_SUPPLY_C}°C)")
print(f"  Pump power:           {rack_min.total_pump_power_w:.0f} W")
print(f"  Max junction temp:    {rack_min.max_junction_temp_c:.1f}°C  (limit: {B200_TJ_LIMIT_C}°C)")

In [ ]:
# Sweep: how does the CDU spec change with supply temperature?
# CDU supply temp is the facility variable operators can control.

supply_temps = [20, 25, 30, 35, 40]
min_flows, dps, returns, pump_powers = [], [], [], []

for t_supply in supply_temps:
    q_min, _ = optimize_flow(OptimizeFlowRateInput(
        heat_load_w=B200_TDP_W, max_junction_temp_c=B200_TJ_LIMIT_C,
        inlet_temp_c=float(t_supply), coolant="water",
        r_jc_k_per_w=B200_R_JC, r_tim_k_per_w=B200_R_TIM, geometry=B200_PLATE,
    ))
    r = analyze_rack(AnalyzeRackInput(
        gpu_count=NVL72_GPU_COUNT, topology="parallel",
        heat_load_per_gpu_w=B200_TDP_W, total_flow_lpm=q_min * NVL72_GPU_COUNT,
        cdu_supply_temp_c=float(t_supply), coolant="water",
        r_jc_k_per_w=B200_R_JC, r_tim_k_per_w=B200_R_TIM, geometry=B200_PLATE,
    ))
    min_flows.append(q_min * NVL72_GPU_COUNT)
    dps.append(r.total_pressure_drop_pa / 1e5)
    returns.append(r.cdu_outlet_temp_c)
    pump_powers.append(r.total_pump_power_w)

print(f"{'Supply (°C)':>12} {'Min Flow (LPM)':>16} {'ΔP (bar)':>10} {'Return (°C)':>12} {'Pump (W)':>10}")
print("-" * 62)
for i, t in enumerate(supply_temps):
    print(f"{t:>12} {min_flows[i]:>16.0f} {dps[i]:>10.3f} {returns[i]:>12.1f} {pump_powers[i]:>10.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(supply_temps, min_flows, 'b-o', linewidth=2, markersize=6)
ax1.set_xlabel('CDU supply temperature (°C)')
ax1.set_ylabel('Minimum total rack flow (LPM)')
ax1.set_title('NVL72 — Required CDU flow vs. supply temp')

ax2.plot(supply_temps, pump_powers, 'r-o', linewidth=2, markersize=6)
ax2.set_xlabel('CDU supply temperature (°C)')
ax2.set_ylabel('Pump power (W)')
ax2.set_title('NVL72 — Pump power at minimum flow')

plt.tight_layout()
plt.show()

---
## Section 2: Series vs. Parallel Topology — What Actually Changes

### The problem

In a **series loop**, coolant flows through GPU 1, then GPU 2, ... GPU N in sequence.
Each GPU's outlet is the next GPU's inlet. Simplest plumbing.

In a **parallel manifold**, all GPUs share the same CDU supply temperature.
Each cold plate is a separate branch. More plumbing, but thermally uniform.

### Important note on how the model represents these topologies

In this model:
- **Series** `total_flow_lpm` = the flow rate through the loop (same fluid flows through every GPU)
- **Parallel** `total_flow_lpm` = total CDU delivery (split equally, so each GPU gets `total/N`)

A fair comparison holds **per-GPU flow constant**. At 8 LPM/GPU:
- Parallel: `total_flow_lpm = 8 × N`
- Series: `total_flow_lpm = 8` (the loop carries 8 LPM through each GPU in sequence)

### The physics

Series: each GPU adds `ΔT_coolant = Q / (ṁ × cp)` to the coolant. Cumulative rise limits how many
GPUs you can chain — at 72 B200s (1,200 W) at 8 LPM, the coolant temperature rise is
72 × 5.2°C = 374°C (far beyond physical limits). Series topology does not scale to
full NVL72 racks at reasonable flow rates.

We demonstrate the tradeoffs with an **8-GPU H100 system**, where the cumulative rise is
manageable, and then explain the NVL72 implication.

In [ ]:
# 8-GPU H100 cluster — series vs. parallel at same per-GPU flow
# Parallel: total_flow_lpm = 8 GPUs × 8 LPM = 64 LPM
# Series:   total_flow_lpm = 8 LPM (same 8 LPM flows through all 8 GPUs in sequence)
N_GPUS = 8
FLOW_PER_GPU = 8.0  # LPM

r_parallel = analyze_rack(AnalyzeRackInput(
    gpu_count=N_GPUS, topology="parallel",
    heat_load_per_gpu_w=700, total_flow_lpm=FLOW_PER_GPU * N_GPUS,
    cdu_supply_temp_c=25.0, coolant="water",
))

r_series = analyze_rack(AnalyzeRackInput(
    gpu_count=N_GPUS, topology="series",
    heat_load_per_gpu_w=700, total_flow_lpm=FLOW_PER_GPU,
    cdu_supply_temp_c=25.0, coolant="water",
))

print(f"8-GPU H100 cluster at {FLOW_PER_GPU:.0f} LPM/GPU (apples-to-apples)")
print()
print(f"{'Parameter':<38} {'Parallel':>12} {'Series':>12}")
print("-" * 64)
print(f"{'Max junction temp (°C)':<38} {r_parallel.max_junction_temp_c:>12.1f} {r_series.max_junction_temp_c:>12.1f}")
print(f"{'CDU return temp (°C)':<38} {r_parallel.cdu_outlet_temp_c:>12.1f} {r_series.cdu_outlet_temp_c:>12.1f}")
print(f"{'Cold plate ΔP (bar)':<38} {r_parallel.total_pressure_drop_pa/1e5:>12.3f} {r_series.total_pressure_drop_pa/1e5:>12.3f}")
print(f"{'Pump power (W)':<38} {r_parallel.total_pump_power_w:>12.1f} {r_series.total_pump_power_w:>12.1f}")
print(f"{'Tj spread (first→last GPU, °C)':<38} {'0.0':>12} {r_series.per_gpu_junction_temps_c[-1]-r_series.per_gpu_junction_temps_c[0]:>12.2f}")
print()
print(f"Series: GPU 0 Tj = {r_series.per_gpu_junction_temps_c[0]:.1f}°C → "
      f"GPU 7 Tj = {r_series.per_gpu_junction_temps_c[-1]:.1f}°C")
print(f"Series: 83°C limit margin at GPU 7: {83 - r_series.per_gpu_junction_temps_c[-1]:.1f}°C")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Per-GPU Tj comparison
gpu_indices = list(range(N_GPUS))
ax1.plot(gpu_indices, r_series.per_gpu_junction_temps_c, 'r-o', markersize=6,
         linewidth=2, label='Series: per-GPU Tj')
ax1.axhline(r_parallel.max_junction_temp_c, color='b', linestyle='--', linewidth=2,
            label=f'Parallel: uniform Tj = {r_parallel.max_junction_temp_c:.1f}°C')
ax1.axhline(83.0, color='k', linestyle=':', linewidth=1.5, label='83°C throttle limit')
ax1.set_xlabel('GPU index (coolant flow order)')
ax1.set_ylabel('Junction temperature (°C)')
ax1.set_title('8-GPU H100 — Per-GPU Tj: Series vs. Parallel')
ax1.legend()
ax1.set_xticks(gpu_indices)

# ΔP and return temp bar chart
categories = ['Series', 'Parallel']
dp_vals = [r_series.total_pressure_drop_pa/1e5, r_parallel.total_pressure_drop_pa/1e5]
ax2.bar(categories, dp_vals, color=['#c0392b', '#2980b9'], alpha=0.8, width=0.4)
ax2.set_ylabel('Cold plate ΔP (bar)')
ax2.set_title('8-GPU H100 — Cold plate ΔP comparison')
for i, v in enumerate(dp_vals):
    ax2.text(i, v + 0.01, f'{v:.3f} bar', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nKey ratio: Series ΔP is {r_series.total_pressure_drop_pa/r_parallel.total_pressure_drop_pa:.0f}× higher than parallel")
print(f"(Equal pump power, but series puts all pressure drop into a single flow path vs. parallel splits it)")

**Key conclusions (8-GPU H100 at 8 LPM/GPU):**

| | Parallel | Series |
|---|---|---|
| Max Tj | 70.9°C | 79.7°C (+8.8°C) |
| CDU return temp | 26.3°C | 35.1°C |
| Cold plate ΔP | 0.168 bar | 1.344 bar (8×) |
| Pump power | 35.8 W | 35.8 W (equal!) |
| Tj spread GPU 0→7 | 0°C | 8.8°C |

**Pump power is equal** — same work (Q×ΔP) regardless of topology, just delivered differently.
Series: low flow through high ΔP. Parallel: high flow through low ΔP.

**Why series doesn't scale to NVL72:** At 72 B200s (1,200 W each) at 8 LPM, the cumulative
coolant temperature rise is 72 × 5.2°C = 374°C — physically impossible. Even at 80 LPM
through the loop, the rise is still 72 × 0.65°C = 47°C, putting the last GPU's inlet at 72°C
before the cold plate adds any resistance. Series loops are used in small clusters (≤ 4–8 GPUs)
where the cumulative rise stays within facility chilled water range.

> **Model limitation:** No flow maldistribution between parallel branches. Real manifolds
> have ±10–30% flow variation. Branch-to-branch Tj variation is not zero in a real parallel system.

In [ ]:
H100_TDP_W = 700.0
H100_TJ_LIMIT_C = 83.0  # NVIDIA datasheet: throttle onset

# Tj vs. flow rate — the core operating curve
flow_rates = np.linspace(2, 20, 80)
results = [analyze(AnalyzeColdplateInput(
    heat_load_w=H100_TDP_W, flow_rate_lpm=float(q),
    inlet_temp_c=25.0, coolant="water"
)) for q in flow_rates]

tj = [r.junction_temp_c for r in results]
pump_w = [r.pump_power_w for r in results]
regime = [r.regime for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Color by regime
colors = {'laminar': 'steelblue', 'transitional': 'orange', 'turbulent': 'firebrick'}
for i in range(len(flow_rates) - 1):
    ax1.plot(flow_rates[i:i+2], tj[i:i+2], color=colors[regime[i]], linewidth=2.5)

ax1.axhline(H100_TJ_LIMIT_C, color='k', linestyle='--', linewidth=1.5, label=f'{H100_TJ_LIMIT_C}°C throttle limit')
ax1.axhline(H100_TJ_LIMIT_C - 5, color='gray', linestyle=':', linewidth=1, label='5°C design margin')
# Add regime legend patches
from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=l) for l, c in colors.items()]
legend_patches.append(plt.Line2D([0], [0], color='k', linestyle='--', label=f'{H100_TJ_LIMIT_C}°C throttle'))
legend_patches.append(plt.Line2D([0], [0], color='gray', linestyle=':', label='5°C margin'))
ax1.legend(handles=legend_patches, fontsize=9)
ax1.set_xlabel('Flow rate (LPM)')
ax1.set_ylabel('Junction temperature (°C)')
ax1.set_title('H100 SXM — Tj vs. flow rate (colored by regime)')

ax2.plot(flow_rates, pump_w, 'g-', linewidth=2)
ax2.set_xlabel('Flow rate (LPM)')
ax2.set_ylabel('Pump power per cold plate (W)')
ax2.set_title('H100 SXM — Pump power vs. flow rate')

plt.tight_layout()
plt.show()

In [ ]:
# Minimum flow at bare limit, and with margins
for margin_c in [0, 3, 5, 8]:
    target = H100_TJ_LIMIT_C - margin_c
    q_opt, analysis_opt = optimize_flow(OptimizeFlowRateInput(
        heat_load_w=H100_TDP_W, max_junction_temp_c=target,
        inlet_temp_c=25.0, coolant="water"
    ))
    print(f"Margin = {margin_c}°C (target {target}°C): "
          f"min flow = {q_opt:.1f} LPM, "
          f"Tj = {analysis_opt.junction_temp_c:.1f}°C, "
          f"pump = {analysis_opt.pump_power_w:.2f} W, "
          f"regime = {analysis_opt.regime}")

In [ ]:
# Manufacturing variation: R_jc ± 20% effect on Tj
# At 8 LPM (typical operating point), how much does R_jc spread affect Tj?

FLOW_OPERATING = 8.0  # LPM — typical H100 operating point
R_JC_NOMINAL = 0.04   # K/W — H100 SXM default

r_nominal = analyze(AnalyzeColdplateInput(
    heat_load_w=H100_TDP_W, flow_rate_lpm=FLOW_OPERATING,
    inlet_temp_c=25.0, coolant="water"
))

# R_jc enters linearly: ΔTj = Q × ΔR_jc
print("R_jc variation across GPU population (8 LPM, 700 W, 25°C inlet):")
print()
for pct in [-20, -10, 0, +10, +20]:
    r_jc = R_JC_NOMINAL * (1 + pct/100)
    tj_adj = r_nominal.junction_temp_c + H100_TDP_W * (r_jc - R_JC_NOMINAL)
    margin = H100_TJ_LIMIT_C - tj_adj
    print(f"  R_jc = {r_jc:.4f} K/W ({pct:+d}%): Tj = {tj_adj:.1f}°C  "
          f"margin = {margin:.1f}°C {'✓' if margin > 0 else '✗ THROTTLE'}")

print()
spread = H100_TDP_W * R_JC_NOMINAL * 0.40  # ±20% = 40% total spread
print(f"Total Tj spread across population: {spread:.1f}°C")
print(f"Design implication: the 12.1°C nominal margin includes real production variance.")
print(f"A GPU at +20% R_jc has only {H100_TJ_LIMIT_C - (r_nominal.junction_temp_c + H100_TDP_W * R_JC_NOMINAL * 0.20):.1f}°C margin.")

**What this means for system design:**

The model gives Tj = 70.9°C at 8 LPM for H100. That 12.1°C margin is not all available margin — it has to cover:
- ±20% R_jc manufacturing spread across 8 (or 72) GPUs: consumes ~5.6°C
- TIM degradation over 2–3 years: consumes another 3–8°C
- Inlet temperature variation (facility chilled water isn't constant): 1–3°C
- Flow rate uncertainty from pump tolerance: 1–2°C

A conservative budget leaves 0–2°C of actual operating margin at nominal flow. **This is why flow optimization results shouldn't be used at bare-limit targets without a margin adder.**

---
## Summary

| Section | Key result |
|---------|------------|
| NVL72 CDU sizing | Minimum flow = 9.3 LPM/GPU (671 LPM total) at 75°C limit, 25°C supply. Every degree colder supply ≈ 10–15% less required flow. |
| Series vs. parallel | Parallel: lower max Tj, lower ΔP, lower pump power; more plumbing. Series: simpler plumbing, large Tj gradient across GPUs, N× ΔP. NVL72 uses parallel. |
| Flow optimization + margin | 12.1°C nominal margin at 8 LPM. After accounting for R_jc manufacturing spread and TIM aging, real margin may be <3°C. Design to Tj_nominal ≤ Tj_limit − 5°C. |

**Next steps:**
- `examples/nvl72_rack_analysis.py` — full procurement analysis with inlet temperature sensitivity sweep
- `examples/rack_topology_tradeoffs.py` — detailed series vs. parallel comparison at multiple flow rates
- `examples/quickstart.py` — minimal local quickstart
- `examples/rack_sizing_example.py` — concise rack sizing comparison